# RhyME quick start

This notebook installs the current local **RhyME** source into a temporary session library and runs it on the bundled longitudinal methylation example. Run the cells from top to bottom with the **R 4.6.1 (Bioconductor 3.23)** kernel.

It contains three checks:

1. a fast 100-CpG fit without a prior, including the current link- and response-scale outputs;
2. a 100-CpG fit with individual-level `sex` and `age` adjustment covariates;
3. the complete 1,000-CpG workflow with the robust empirical-Bayes prior and default joint Wald test.

In [ ]:
source_candidates <- unique(normalizePath(
    c(getwd(), file.path(getwd(), "RhyME"), dirname(getwd())),
    mustWork = FALSE
))
is_package <- file.exists(file.path(source_candidates, "DESCRIPTION"))
if (!any(is_package)) {
    stop("Run this notebook from the RhyME repository or its notebooks directory.")
}
PACKAGE_DIR <- source_candidates[which(is_package)[1L]]
R_BIN <- normalizePath(file.path(R.home("bin"), "R"))
R_PREFIX_BIN <- normalizePath(
    file.path(R.home(), "..", "..", "bin"), mustWork = FALSE
)
Sys.setenv(PATH = paste(R_PREFIX_BIN, dirname(R_BIN), Sys.getenv("PATH"),
                       sep = .Platform$path.sep))
LOCAL_LIB <- tempfile("RhyME-notebook-lib-")
dir.create(LOCAL_LIB)
INSTALL_LOG <- file.path(tempdir(), "RhyME_quickstart_install.log")
install_status <- system2(
    R_BIN, c("CMD", "INSTALL", paste0("--library=", LOCAL_LIB),
             PACKAGE_DIR), stdout = INSTALL_LOG, stderr = INSTALL_LOG
)
if (install_status != 0L) {
    cat(paste(readLines(INSTALL_LOG, warn = FALSE), collapse = "\n"))
    stop("Local RhyME installation failed; inspect ", INSTALL_LOG)
}
.libPaths(c(LOCAL_LIB, .libPaths()))
required <- c("RhyME", "TMB", "SummarizedExperiment", "BiocParallel")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) stop("Missing package(s): ", paste(missing, collapse = ", "))

data.frame(
    R = R.version.string,
    RhyME = as.character(packageVersion("RhyME")),
    TMB = as.character(packageVersion("TMB")),
    library = find.package("RhyME")
)

In [ ]:
suppressPackageStartupMessages({
    library(RhyME)
    library(SummarizedExperiment)
    library(BiocParallel)
})

## Load the bundled example

The assay contains CpGs in rows and samples in columns. `colData(se)` contains the sampling time and subject identifier used by the mixed model.

In [ ]:
example_file <- system.file("extdata", "example_data.rds", package = "RhyME")
stopifnot(nzchar(example_file))
example <- readRDS(example_file)
se <- example$se

data.frame(
    CpGs = nrow(assay(se)),
    samples = ncol(assay(se)),
    subjects = length(unique(colData(se)$subject)),
    time_points = length(unique(colData(se)$time))
)
head(as.data.frame(colData(se)))

## 1. Fast smoke test

This first run checks model fitting and output without estimating a genome-wide prior. It should finish quickly.

In [ ]:
quick_se <- se[seq_len(100), ]
quick_fit <- fitCosinor(
    quick_se,
    assay_name = "beta",
    time_col = "time",
    subject_col = "subject",
    prior = "none",
    test = "Wald",
    BPPARAM = SerialParam(),
    verbose = TRUE
)
quick_results <- rowData(quick_fit)
expected_columns <- c(
    "amplitude_link", "amplitude_response", "phase",
    "mesor_link", "mesor_response", "tau",
    "pvalue", "padj", "converged"
)
stopifnot(all(expected_columns %in% names(quick_results)))
list(
    fit_summary = metadata(quick_fit)$RhyME$fitSummary,
    returned_columns = expected_columns
)

In [ ]:
quick_top <- topCpGs(quick_fit, n = 10)
as.data.frame(quick_top)

In [ ]:
plotCosinorFit(
    quick_fit,
    cpgs = as.character(quick_top$cpg[seq_len(min(6, nrow(quick_top)))]),
    ncol = 3
)

## 2. Individual-level covariate smoke test

The bundled data do not contain `sex` or `age`, so this cell creates one fixed value per subject and repeats it across that subject's observations. These variables adjust both rhythmic and null models; the returned p-value still tests only sine and cosine.

In [ ]:
cov_se <- quick_se
cd <- as.data.frame(colData(cov_se))
subject_ids <- unique(as.character(cd$subject))
subject_data <- data.frame(
    subject = subject_ids,
    sex = rep(c("F", "M"), length.out = length(subject_ids)),
    age = seq(40, by = 3, length.out = length(subject_ids))
)
cd$sex <- subject_data$sex[match(cd$subject, subject_data$subject)]
cd$age <- subject_data$age[match(cd$subject, subject_data$subject)]
colData(cov_se) <- S4Vectors::DataFrame(cd)
cov_fit <- fitCosinor(
    cov_se, time_col = "time", subject_col = "subject",
    covariates = c("sex", "age"), prior = "none",
    test = "Wald", BPPARAM = SerialParam(), verbose = TRUE
)
stopifnot(identical(
    metadata(cov_fit)$RhyME$design$covariates, c("sex", "age")
))
metadata(cov_fit)$RhyME$fitSummary

## 3. Complete automatic-prior workflow

This is the intended genome-wide path. RhyME first fits at least 1,000 pilot CpGs without a prior, estimates the center and scale of the log-precision distribution using the median and normalized IQR, and then performs the final fit. The example below uses 1,000 CpGs so it may take a few minutes.

In [ ]:
N_AUTO <- 1000L
WORKERS <- min(4L, max(1L, parallel::detectCores(logical = FALSE)))
bp <- if (.Platform$OS.type == "windows") {
    SnowParam(workers = WORKERS, RNGseed = 20260820, progressbar = FALSE)
} else {
    MulticoreParam(workers = WORKERS, RNGseed = 20260820, progressbar = FALSE)
}
data.frame(CpGs = N_AUTO, workers = WORKERS, test = "Wald")

In [ ]:
set.seed(20260820)
auto_se <- se[seq_len(N_AUTO), ]
elapsed <- system.time({
    auto_fit <- fitCosinor(
        auto_se,
        assay_name = "beta",
        time_col = "time",
        subject_col = "subject",
        prior = "auto",
        test = "Wald",
        BPPARAM = bp,
        verbose = TRUE
    )
})

auto_meta <- metadata(auto_fit)$RhyME
data.frame(
    prior_mean_log_phi = auto_meta$ebPrior$mean,
    prior_sd_log_phi = auto_meta$ebPrior$sd,
    converged = auto_meta$fitSummary$converged,
    total = auto_meta$fitSummary$total,
    elapsed_minutes = unname(elapsed[["elapsed"]]) / 60
)

In [ ]:
auto_top <- topCpGs(auto_fit, n = 20)
as.data.frame(auto_top)

In [ ]:
plotCosinorFit(
    auto_fit,
    cpgs = as.character(auto_top$cpg[seq_len(min(6, nrow(auto_top)))]),
    ncol = 3
)

## Use your own data

Your beta-value matrix must have CpGs in rows and samples in columns. The sample metadata rows must follow the same sample order and contain numeric collection time plus a subject ID for repeated measurements.

In [ ]:
# Template (replace beta_matrix and sample_metadata with your objects):
# stopifnot(identical(colnames(beta_matrix), rownames(sample_metadata)))
# user_se <- SummarizedExperiment(
#     assays = list(beta = beta_matrix),
#     colData = S4Vectors::DataFrame(sample_metadata)
# )
# user_fit <- fitCosinor(
#     user_se, time_col = "time", subject_col = "subject",
#     prior = "auto", test = "Wald", BPPARAM = bp
# )
# write.csv(as.data.frame(rowData(user_fit)), "RhyME_results.csv")
# saveRDS(user_fit, "RhyME_fit.rds")

cat("Template ready. Uncomment it after loading your own matrix and metadata.\n")